# Detecting Tomato **Wilt** Symptoms with YOLO Object Detection


### What will the model do?
This notebook trains an **object detection** model. For each photo, the model must find
**where** the symptomatic plant parts are, by drawing a **bounding box**, and say **what** each
box contains (its **class**). The model reports *how many* symptomatic
organs there are and *where* they are on the plant.

### The data
We use the `Wilt` subset of the **Tomato Disease Dataset**: **527 photos** with
**1373 hand-drawn bounding boxes**. The boxes were drawn by annotators and are stored in
**Pascal-VOC XML** files. Each box is labelled with the plant organ or region that shows symptoms:

| Class | What is boxed | Boxes |
|---|---|---|
| `Wilt_Leaf`   | Wilted leaves | 405 |
| `Wilt_Middle` | Wilted mid-canopy region | 345 |
| `Wilt_Top`    | Wilted shoot tip | 298 |
| `Wilt_Base`   | Base of the plant | 173 |
| `Wilt_Stem`   | Affected stem | 135 |
| `Viral_Leaf`  | Leaves labelled as *viral* (7 images only) | 17 |

Things to keep in mind:
- The annotations include **17 `Viral_Leaf` boxes in 7 Wilt images**. These may be genuine mixed
  symptoms or labelling inconsistencies. We keep them as-is, but they form a tiny 6th class that
  the model will struggle with. Mention this in your discussion.
- The classes describe **plant organs/regions**, which overlap (e.g. a `Wilt_Leaf` box often sits
  inside a `Wilt_Middle` box), so this is a hard detection problem.
- 48 of the 527 photos carry an EXIF rotation flag (see Step 3).

### Learning objectives
By the end of this notebook you should be able to:
1. Explain how annotated images are prepared for training (format conversion, train/val/test split).
2. Train a YOLO detector using **transfer learning** from pretrained weights.
3. Evaluate a detector with **precision, recall and mAP**, overall and per class.
4. Run a small experiment (change one setting, compare results) and interpret the outcome.

### Roadmap
| Step | What happens |
|---|---|
| 1 | Set up Colab (GPU, libraries) |
| 2 | Download the dataset |
| 3 | Convert annotations to YOLO format and split into train / validation / test |
| 4 | Train the detector (**settings you can change are here**) |
| 5 | Inspect the training curves |
| 6 | Evaluate on the validation set |
| 7 | Evaluate on the **test** set (your final numbers) |
| 8 | Predict on test images and analyse the detections |
| 9 | Log the experiment |
| Final | Instructions for your report |

> Throughout the notebook, notes marked **For your report** tell you what to write down.

## Key concepts

**YOLO (You Only Look Once)** is a family of fast object detectors. The image passes through a
convolutional neural network **once**, and the network directly outputs a set of boxes, each with
a class and a **confidence score** (0-1). We use the **Ultralytics** implementation
(https://docs.ultralytics.com).

| Term | Meaning |
|---|---|
| **Bounding box** | Rectangle around an object: `(x_center, y_center, width, height)` |
| **IoU** (Intersection over Union) | Overlap between a predicted and a true box (0 = none, 1 = perfect). A prediction counts as correct (*true positive*) when IoU ≥ a threshold, e.g. 0.5 |
| **Precision** | Of all boxes the model predicted, the fraction that are correct: *"how many false alarms?"* |
| **Recall** | Of all true boxes, the fraction the model found: *"how many did it miss?"* |
| **AP** (Average Precision) | Area under the precision-recall curve for one class (0-1) |
| **mAP50** | AP averaged over all classes, using IoU ≥ 0.50. The headline metric |
| **mAP50-95** | mAP averaged over IoU thresholds 0.50…0.95. Stricter: rewards precise box placement |
| **Transfer learning** | Start from a model already trained on a large dataset (COCO, 80 everyday object classes) and fine-tune it on our images. Essential with small datasets |
| **Epoch** | One full pass through the training images |
| **Train / validation / test** | *Train*: the model learns from these. *Validation*: used during training to pick the best epoch. *Test*: never seen during training, so it gives an unbiased final score |

# 1. Setup

### 1.1 Turn on a GPU
In Colab, open `Runtime > Change runtime type`, choose **T4 GPU** (or any GPU) and click `Save`.
Training on a CPU works, but it is roughly 10-50× slower.

### 1.2 Install and import
The next cells check the GPU, install the **Ultralytics** library (which provides YOLO) and
import the Python packages used in the notebook. If `nvidia-smi` prints an error, you are not on
a GPU runtime.

In [ ]:
# Check GPU access (should list a GPU such as a Tesla T4)
!nvidia-smi

**Install Ultralytics**, the library that provides YOLO. `-U` upgrades it to the latest version,
so the newest YOLO models (YOLO26) are available.

In [ ]:
!pip install -q -U ultralytics

**Import the libraries** used in the notebook and choose the device: GPU 0 if one is available,
otherwise the CPU. The cell prints the device that will be used for training.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os, glob, random, shutil
from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET
import numpy as np
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"   # GPU 0 if available, else CPU
print("Ultralytics YOLO loaded. Training device:", DEVICE)

# 2. Get the data

The photos come from the **Tomato Disease Dataset** on Mendeley Data:
https://data.mendeley.com/datasets/c2x8rynybg/1 (open this page to read the description of the
dataset).

You do not need to download anything yourself: the download link
(`https://data.mendeley.com/public-api/zip/c2x8rynybg/download/1`) is already in the cell below.
The cell downloads the archive (about 3 GB, a few minutes) into Colab and unzips it.

It then finds the dataset root (the folder containing
`images/` and `annotations/`) and prints its location.
Inside, each disease has its own sub-folder; this notebook uses only `images/Wilt` and
`annotations/Wilt`.

In [ ]:
# ---- Paste your download link here ----
DATA_URL = "https://data.mendeley.com/public-api/zip/c2x8rynybg/download/1"   # Google Drive share link OR a direct URL
ARCHIVE  = "dataset.zip"                      # local filename to save the download as

!pip install -q gdown
os.makedirs("/content/data", exist_ok=True)

# Download: gdown parses Google Drive share links (--fuzzy); wget handles any direct URL
if "drive.google.com" in DATA_URL or "docs.google.com" in DATA_URL:
    !gdown --fuzzy "{DATA_URL}" -O {ARCHIVE}
else:
    !wget -q --show-progress -O {ARCHIVE} "{DATA_URL}"

# Unzip if it is a zip archive
if ARCHIVE.lower().endswith(".zip"):
    !unzip -qo {ARCHIVE} -d /content/data
else:
    !cp {ARCHIVE} /content/data/ 2>/dev/null

# Auto-detect the dataset root = folder that contains images/ and annotations/
# (skip the __MACOSX folder that zips made on a Mac often contain)
cands = sorted(os.path.dirname(p) for p in glob.glob("/content/data/**/annotations", recursive=True)
               if "__MACOSX" not in p)
assert cands, "No 'annotations' folder found - check DATA_URL and the unzip output above."
DATA_ROOT = cands[0]
print("DATA_ROOT =", DATA_ROOT)
!ls "{DATA_ROOT}"

# 3. Prepare the data for YOLO

YOLO cannot read Pascal-VOC XML directly. It expects **one `.txt` label file per image**, with
one line per box:

```
<class_id> <x_center> <y_center> <width> <height>      # all values normalised to 0-1
```

This step has five parts:
1. **Discover the classes** and count the boxes per class.
2. **Split** the images into train (70 %), validation (20 %) and test (10 %).
3. **Convert** every XML file to a YOLO `.txt` file and copy the image into its split folder.
4. Write the **`data.yaml`** file that tells YOLO where the data is and what the classes are.
5. **Visualise** a converted image to check that the boxes still line up.

### 3.1 Discover classes
We read every XML file in `annotations/Wilt`, collect the class names, fix the known
`Virus_Middle` into `Viral_Middle` typo, and give each class an integer ID (alphabetical order).

> **For your report (Methodology):** record the number of images, the classes and the number
> of boxes per class. Is the dataset balanced?

In [ ]:
# ---- Paths: restricted to the Wilt subset (DATA_ROOT set in the download cell) ----
CATEGORY  = "Wilt"
IMG_ROOT  = os.path.join(DATA_ROOT, "images", CATEGORY)
ANN_ROOT  = os.path.join(DATA_ROOT, "annotations", CATEGORY)
OUT_ROOT  = "/content/tomato_wilt_yolo"

NAME_FIXES = {"Virus_Middle": "Viral_Middle"}   # fix annotation typo
IMG_EXTS   = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

def find_image_for_xml(xml_path):
    """Locate the image matching an annotation file (same Wilt folder)."""
    stem = Path(xml_path).stem
    for ext in IMG_EXTS:
        p = os.path.join(IMG_ROOT, stem + ext)
        if os.path.exists(p):
            return p
    for ext in IMG_EXTS:
        hits = sorted(glob.glob(os.path.join(IMG_ROOT, "**", stem + ext), recursive=True))
        if hits:
            return hits[0]
    return None

def class_name(obj):
    nm = obj.find("name").text.strip()
    return NAME_FIXES.get(nm, nm)

# Sorted so the file order (and therefore the random split) is identical on every machine
all_xml = sorted(glob.glob(os.path.join(ANN_ROOT, "**", "*.xml"), recursive=True))
assert all_xml, f"No XML files found in {ANN_ROOT}"

box_counts = Counter(class_name(obj) for x in all_xml
                     for obj in ET.parse(x).getroot().findall("object"))
classes = sorted(box_counts)
class_to_id = {c: i for i, c in enumerate(classes)}

print(f"{len(all_xml)} Wilt annotation files | {sum(box_counts.values())} boxes | {len(classes)} classes:")
for c in classes:
    print(f"  {class_to_id[c]:>2}  {c:15s} {box_counts[c]:>5} boxes")

### 3.2 Train / validation / test split
Each XML file is paired with its image, the pairs are shuffled with a **fixed random seed
(42)** so everyone gets the same split, and then cut 70 / 20 / 10.

> **Note:** The split is random *per image*. Photos taken on the same day of the same plant can end up
> in different splits, which can make the test score look better than it really is. Keep this
> in mind for the discussion.

In [ ]:
# Pair each annotation with its image, then shuffle & split
random.seed(42)
SPLITS = {"train": 0.70, "valid": 0.20, "test": 0.10}

pairs, missing = [], 0
for x in all_xml:
    img = find_image_for_xml(x)
    if img is None:
        missing += 1
        continue
    pairs.append((img, x))
print(f"Matched {len(pairs)} image/annotation pairs ({missing} XMLs had no image).")

stems = Counter(Path(img).stem for img, _ in pairs)
assert max(stems.values()) == 1, "Duplicate image names would overwrite each other in the output."

random.shuffle(pairs)
n = len(pairs)
n_train = int(n * SPLITS["train"])
n_valid = int(n * SPLITS["valid"])
split_of = {}
for i, pr in enumerate(pairs):
    split_of[pr] = "train" if i < n_train else ("valid" if i < n_train + n_valid else "test")

# start from empty folders so re-running this cell never mixes old and new splits
shutil.rmtree(OUT_ROOT, ignore_errors=True)
for s in SPLITS:
    os.makedirs(os.path.join(OUT_ROOT, s, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, s, "labels"), exist_ok=True)
print("Split sizes:", dict(Counter(split_of.values())))

### 3.3 Convert boxes to YOLO format (with EXIF orientation fix)

For each image we read its boxes from the XML file, convert the corner coordinates
`(xmin, ymin, xmax, ymax)` in pixels into normalised `(x_center, y_center, width, height)`, and
write them to a `.txt` file next to a copy of the image.

> **Why the EXIF fix matters.** Many of these phone photos are stored *sideways*, with an
> EXIF "orientation" flag telling viewers to rotate them (e.g. "rotate 90°"). The boxes in the
> XML files were drawn on the **stored (un-rotated) pixels**, but most image viewers and some
> loaders apply the flag, so the boxes would no longer line up with the plant. To make the data
> consistent everywhere, we **rotate the pixels permanently** (`exif_transpose`), remove the
> flag, and **transform the box coordinates the same way**. Without this step the model would
> learn from boxes in the wrong place.

In [ ]:
# NOTE: import PIL's Image as PILImage so we don't shadow IPython.display.Image,
# which the display cells below rely on (Image(filename=...)).
from PIL import Image as PILImage, ImageOps, ExifTags
PILImage.MAX_IMAGE_PIXELS = None
ORIENT_TAG = next(k for k, v in ExifTags.TAGS.items() if v == "Orientation")

def transpose_box(x1, y1, x2, y2, W, H, o):
    """Map a box from raw-pixel space to exif_transpose'd space. Returns box + new (W,H)."""
    if o == 2:                       # mirror left-right
        return (W - x2, y1, W - x1, y2, W, H)
    if o == 3:                       # 180 deg
        return (W - x2, H - y2, W - x1, H - y1, W, H)
    if o == 4:                       # mirror top-bottom
        return (x1, H - y2, x2, H - y1, W, H)
    if o == 5:                       # transpose      -> display becomes H x W
        return (y1, x1, y2, x2, H, W)
    if o == 6:                       # 90 deg CW      -> display becomes H x W
        return (H - y2, x1, H - y1, x2, H, W)
    if o == 7:                       # transverse     -> display becomes H x W
        return (H - y2, W - x2, H - y1, W - x1, H, W)
    if o == 8:                       # 90 deg CCW     -> display becomes H x W
        return (y1, W - x2, y2, W - x1, H, W)
    return (x1, y1, x2, y2, W, H)    # 1 = normal

def clamp(v):
    return min(max(v, 0.0), 1.0)

n_rotated = 0
def convert_one(img_path, xml_path, split):
    global n_rotated
    root = ET.parse(xml_path).getroot()

    im = PILImage.open(img_path)
    W, H = im.size                                           # stored-pixel size (= XML <size>)
    o  = im.getexif().get(ORIENT_TAG, 1)
    corrected = ImageOps.exif_transpose(im).convert("RGB")   # bake rotation, drop flag
    if o != 1:
        n_rotated += 1

    lines = []
    for obj in root.findall("object"):
        cid = class_to_id[class_name(obj)]
        b = obj.find("bndbox")
        x1, y1 = float(b.find("xmin").text), float(b.find("ymin").text)
        x2, y2 = float(b.find("xmax").text), float(b.find("ymax").text)
        nx1, ny1, nx2, ny2, nW, nH = transpose_box(x1, y1, x2, y2, W, H, o)
        assert (nW, nH) == corrected.size, f"orientation mismatch in {img_path}"
        xc, yc = ((nx1 + nx2) / 2) / nW, ((ny1 + ny2) / 2) / nH
        bw, bh = (nx2 - nx1) / nW, (ny2 - ny1) / nH
        lines.append(f"{cid} {clamp(xc):.6f} {clamp(yc):.6f} {clamp(bw):.6f} {clamp(bh):.6f}")

    # save the EXIF-corrected image (no orientation flag) so every loader agrees
    stem = Path(img_path).stem
    corrected.save(os.path.join(OUT_ROOT, split, "images", stem + ".jpg"), quality=95)
    with open(os.path.join(OUT_ROOT, split, "labels", stem + ".txt"), "w") as f:
        f.write("\n".join(lines))

for (img, xml), s in split_of.items():
    convert_one(img, xml, s)

# boxes per class in each split (useful table for the report)
import pandas as pd
split_box_counts = {s: Counter() for s in SPLITS}
for s in SPLITS:
    for lbl in glob.glob(os.path.join(OUT_ROOT, s, "labels", "*.txt")):
        for line in open(lbl):
            if line.strip():
                split_box_counts[s][classes[int(line.split()[0])]] += 1
split_table = pd.DataFrame(split_box_counts).reindex(classes).fillna(0).astype(int)
split_table.loc["images"] = [len(glob.glob(os.path.join(OUT_ROOT, s, "images", "*"))) for s in SPLITS]

print(f"Baked EXIF rotation into {n_rotated} images (boxes transformed to match).")
split_table

> **For your report (Methodology):** the table above (boxes per class in each split) is a good
> dataset table. Do the rare classes have enough examples in the *test* split for a reliable
> score?

### 3.4 Write `data.yaml`
YOLO reads one small configuration file that lists the folders of each split and the class names.

In [ ]:
import yaml
data_yaml = {
    "path":  OUT_ROOT,
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    len(classes),
    "names": classes,
}
YAML_PATH = os.path.join(OUT_ROOT, "data.yaml")
with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(open(YAML_PATH).read())

### 3.5 Visual check of the converted labels
**Always look at your data.** The cell below draws the converted boxes on a few random training
images. The boxes should sit tightly around symptomatic organs. If they look shifted or
rotated, something went wrong in the conversion. Re-run the cell to see other images.

In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

def draw_yolo_boxes(ax, image_path, label_path, class_names):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(label_path):
        for line in open(label_path):
            p = line.split()
            if len(p) != 5:
                continue
            cid = int(p[0]); xc, yc, bw, bh = map(float, p[1:])
            x1, y1 = int((xc - bw/2)*w), int((yc - bh/2)*h)
            x2, y2 = int((xc + bw/2)*w), int((yc + bh/2)*h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), max(2, w // 300))
            cv2.putText(img, class_names[cid], (x1, max(y1 - 10, 30)),
                        cv2.FONT_HERSHEY_SIMPLEX, w / 1500, (255, 0, 0), max(2, w // 500))
    ax.imshow(img); ax.set_title(Path(image_path).name, fontsize=9); ax.axis("off")

train_imgs = sorted(glob.glob(os.path.join(OUT_ROOT, "train", "images", "*")))
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, sample in zip(axes, random.sample(train_imgs, 3)):
    label = os.path.join(OUT_ROOT, "train", "labels", Path(sample).stem + ".txt")
    draw_yolo_boxes(ax, sample, label, classes)
plt.tight_layout(); plt.show()

# 4. Train the Wilt detector

### 4.1 Experiment settings: **this is the cell you change**

Training starts from YOLO weights **pretrained on the COCO dataset** (transfer learning). The
network already knows general visual features such as edges, textures and shapes, and only has
to learn what wilt symptoms look like. This matters with a dataset as small as ours.

| Setting | What it does | Suggested values to try |
|---|---|---|
| `MODEL_NAME` | Model size: **n**ano < **s**mall < **m**edium < **l**arge. Bigger = more capacity, slower, needs more data | `yolo26n.pt` (default), `yolo26s.pt`, `yolo26m.pt` |
| `EPOCHS` | Maximum number of passes over the training data | 50, 100 (default), 200 |
| `PATIENCE` | Early stopping: stop if validation mAP has not improved for this many epochs | 20 (default), 50 |
| `IMGSZ` | Images are resized to this many pixels. Larger helps small objects but is slower and uses more memory | 640 (default), 960 |
| `BATCH` | Images per training step. Lower it if you get a *CUDA out of memory* error | 16 (default), 8 |

> **Do a proper experiment:** change **one** setting at a time and compare against your
> baseline. Each run gets its own name (`RUN_NAME`) and folder, so results never overwrite each
> other.
>
> On a Colab T4 GPU the default run takes roughly 5-20 minutes, depending on the subset.

In [ ]:
# ======================= EXPERIMENT SETTINGS (edit me) =======================
MODEL_NAME = "yolo26n.pt"   # yolo26n.pt / yolo26s.pt / yolo26m.pt  (or yolo11n.pt ...)
EPOCHS     = 100
PATIENCE   = 20
IMGSZ      = 640
BATCH      = 16
# ==============================================================================
RUN_NAME   = f"{CATEGORY}_{Path(MODEL_NAME).stem}_e{EPOCHS}_img{IMGSZ}_b{BATCH}"
PROJECT    = "/content/runs/wilt"
print("Run name:", RUN_NAME)

### 4.2 Train
**Check the installation.** `yolo checks` prints the software and hardware versions (useful for the *Methodology*).

In [ ]:
!yolo checks

**Train the model.** The cell loads the pretrained weights given in `MODEL_NAME`. If that model is not available in the installed
Ultralytics version, it falls back to the equivalent YOLO11 model and says so. It then trains with the
settings from the experiment-settings cell. The
training cell then prints one line per epoch with the losses and the validation precision (`P`),
recall (`R`), `mAP50` and `mAP50-95`.

The model keeps two checkpoints: `last.pt` (final epoch) and **`best.pt`** (the epoch with the
best validation score). We use `best.pt` from here on.

In [ ]:
try:
    model = YOLO(MODEL_NAME)
except Exception as e:                       # YOLO26 needs a recent ultralytics version
    fallback = MODEL_NAME.replace("yolo26", "yolo11")
    print(f"'{MODEL_NAME}' unavailable ({e}); falling back to {fallback}")
    MODEL_NAME = fallback
    model = YOLO(MODEL_NAME)

train_results = model.train(
    data=YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,   # early-stop if no improvement for PATIENCE epochs
    device=DEVICE,       # GPU 0 if available, else "cpu"
    seed=42,
    project=PROJECT,
    name=RUN_NAME,       # each experiment gets its own folder
)

RUN_DIR = Path(model.trainer.save_dir)        # actual folder used (e.g. ..._2 if the name existed)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print("Run folder:", RUN_DIR)

# 5. Training curves

`results.png` plots the training history, one curve per epoch:
- **`train/box_loss`, `train/cls_loss`, `train/dfl_loss`**: how wrong the box positions and
  class predictions are on the *training* images. These should go down.
- **`val/...` losses**: the same on the *validation* images. If the training losses keep
  falling while the validation losses rise, the model is **over-fitting** (memorising the training
  images instead of learning general features).
- **`metrics/precision`, `recall`, `mAP50`, `mAP50-95`**: validation scores. These should rise
  and level off.

> **For your report (Results):** include this figure for your best run. Did the curves level
> off, or would more epochs have helped? At which epoch did training stop, and was it early stopping?

In [ ]:
results_img = RUN_DIR / "results.png"
if results_img.exists():
    display(Image(filename=str(results_img), width=1000))
else:
    print("results.png not found in", RUN_DIR)

import pandas as pd
hist = pd.read_csv(RUN_DIR / "results.csv")
hist.columns = hist.columns.str.strip()
best_row = hist.loc[hist["metrics/mAP50(B)"].idxmax()]
print(f"Epochs run: {len(hist)} | best val mAP50 = {best_row['metrics/mAP50(B)']:.3f} at epoch {int(best_row['epoch'])}")

# 6. Evaluate on the validation set

We reload `best.pt` and score it on the **validation** split. The table shows, for each class:
`Images`, `Instances` (true boxes), precision `P`, recall `R`, `mAP50` and `mAP50-95`.

Remember that the validation set was used to choose the best epoch, so this score is slightly
optimistic. The **test** set in Step 7 gives the unbiased number.

In [ ]:
model = YOLO(str(BEST_PT))
val_metrics = model.val(data=YAML_PATH, split="val", imgsz=IMGSZ, device=DEVICE,
                        project=PROJECT, name=f"{RUN_NAME}_val")
print(f"VALIDATION  mAP50 = {val_metrics.box.map50:.3f} | mAP50-95 = {val_metrics.box.map:.3f}")

# 7. Evaluate on the **test** set (final results)

The test images were **never seen during training or model selection**, so these are the
numbers to report as your final performance. We build a per-class table and show the
**confusion matrix** and **precision-recall curve**.

**Reading the confusion matrix:** rows/columns are classes plus `background`. Values on the
diagonal are correct detections. Off-diagonal cells show one class predicted as another.
A *background* column/row means boxes that were **missed** or **false alarms**.

> **For your report (Results):** the per-class table below is your main results table
> (Table 2). Which class is easiest, which is hardest, and how does that relate to its number of
> training boxes (Step 3.3)?

In [ ]:
test_metrics = model.val(data=YAML_PATH, split="test", imgsz=IMGSZ, device=DEVICE,
                         project=PROJECT, name=f"{RUN_NAME}_test")

rows = []
for i, cid in enumerate(test_metrics.box.ap_class_index):
    p, r, ap50, ap = test_metrics.box.class_result(i)
    rows.append({"class": test_metrics.names[int(cid)], "train_boxes": split_box_counts["train"][test_metrics.names[int(cid)]],
                 "test_boxes": split_box_counts["test"][test_metrics.names[int(cid)]],
                 "precision": p, "recall": r, "mAP50": ap50, "mAP50-95": ap})
rows.append({"class": "ALL", "train_boxes": sum(split_box_counts["train"].values()),
             "test_boxes": sum(split_box_counts["test"].values()),
             "precision": test_metrics.box.mp, "recall": test_metrics.box.mr,
             "mAP50": test_metrics.box.map50, "mAP50-95": test_metrics.box.map})
per_class_test = pd.DataFrame(rows).round(3)
per_class_test.to_csv(RUN_DIR / "per_class_test_metrics.csv", index=False)
print("Classes without any test boxes are not listed (they cannot be scored).")
per_class_test

**Confusion matrix and precision-recall curve (test set).** The cell shows the two figures that
Ultralytics saved during the test evaluation: the normalised confusion matrix (read it as
explained above) and the precision-recall curve of each class, with its AP.

In [ ]:
TEST_DIR = Path(test_metrics.save_dir)
for pattern in ["confusion_matrix_normalized.png", "*PR_curve.png"]:
    for f in sorted(TEST_DIR.glob(pattern))[:1]:
        print(f.name); display(Image(filename=str(f), width=700))

# 8. Predict on the test images

### 8.1 Visual inspection
The trained model now predicts boxes on every test image. Each predicted box shows the class
and confidence. Only boxes above the confidence threshold (`CONF`, default 0.25) are kept:
lowering it gives more detections (higher recall, more false alarms) and raising it gives the
opposite.

Compare the predictions with the ground-truth boxes (left vs right). Look for **missed
symptoms**, **false alarms** and **wrong classes**.

> **For your report (Results/Discussion):** save one or two good examples and at least one
> failure. Is the failure a real model error, or is the annotation itself questionable?

In [ ]:
CONF = 0.25   # confidence threshold for keeping a predicted box

test_img_dir = os.path.join(OUT_ROOT, "test", "images")
pred_results = list(model.predict(test_img_dir, conf=CONF, save=True, device=DEVICE, verbose=False,
                                  project=PROJECT, name=f"{RUN_NAME}_test_predictions", stream=True))
PRED_DIR = Path(pred_results[0].save_dir)
print(f"Predicted {len(pred_results)} test images -> annotated copies saved in {PRED_DIR}")

# side-by-side: ground truth (left) vs prediction (right) for a few random test images
for r in random.sample(pred_results, min(3, len(pred_results))):
    stem = Path(r.path).stem
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))
    draw_yolo_boxes(a1, r.path, os.path.join(OUT_ROOT, "test", "labels", stem + ".txt"), classes)
    a1.set_title(f"Ground truth: {stem}")
    a2.imshow(r.plot()[:, :, ::-1]); a2.axis("off")
    a2.set_title(f"Prediction: {len(r.boxes)} boxes")
    plt.tight_layout(); plt.show()

### 8.2 Post-analysis: how many symptomatic organs per image?
In practice a grower wants two answers for each plant photo: **"is the disease there?"** and
**"how many organs are affected?"** (a rough measure of severity). So for every test image we
count the **detected boxes per class** and compare them with the **true number of boxes** in
the annotation.

The summary table reports, per class:
- `imgs_with_class`: test images that truly contain the class.
- `image_hit_rate`: fraction of those images where the model found **at least one** box of
  that class (image-level sensitivity: *"was the symptom noticed?"*).
- `false_alarm_imgs`: images where the model predicted the class although it is not annotated.
- `mean_true` / `mean_pred`: average number of boxes per image (truth vs model).
- `count_MAE`: mean absolute error of the per-image count (0 = the model counts perfectly).
- `count_bias`: mean of (predicted − true). Negative = the model **under-counts** (misses
  organs), positive = it **over-counts** (duplicates or false alarms).

The per-image table is saved as `per_image_counts.csv` and every single detection as
`detection_results.csv` in the run folder.

> **For your report (Results/Discussion):** a model can have a modest mAP and still be useful
> if it reliably *flags the image* and gets the *count* about right. Does your model tend to
> under- or over-count? For which classes?

In [ ]:
det_rows, img_rows = [], []
for r in pred_results:
    stem = Path(r.path).stem
    lbl  = os.path.join(OUT_ROOT, "test", "labels", stem + ".txt")
    true_n = Counter(classes[int(l.split()[0])] for l in open(lbl) if l.strip())
    pred_n = Counter(r.names[int(c)] for c in r.boxes.cls.tolist())
    row = {"image": Path(r.path).name}
    for c in classes:
        row[f"{c}_true"], row[f"{c}_pred"] = true_n[c], pred_n[c]
    row["ALL_true"], row["ALL_pred"] = sum(true_n.values()), sum(pred_n.values())
    img_rows.append(row)
    for b in r.boxes:
        det_rows.append({"image": Path(r.path).name, "class": r.names[int(b.cls)],
                         "conf": round(float(b.conf), 3),
                         **dict(zip(["x1", "y1", "x2", "y2"], [round(v) for v in b.xyxy[0].tolist()]))})

per_image = pd.DataFrame(img_rows)
per_image.to_csv(RUN_DIR / "per_image_counts.csv", index=False)
pd.DataFrame(det_rows, columns=["image", "class", "conf", "x1", "y1", "x2", "y2"]) \
  .to_csv(RUN_DIR / "detection_results.csv", index=False)

summary = []
for c in classes + ["ALL"]:
    t, p = per_image[f"{c}_true"], per_image[f"{c}_pred"]
    present = t > 0
    summary.append({
        "class": c,
        "imgs_with_class":  int(present.sum()),
        "image_hit_rate":   (p[present] > 0).mean() if present.any() else float("nan"),
        "false_alarm_imgs": int(((p > 0) & ~present).sum()),
        "mean_true":        t[present].mean() if present.any() else 0.0,
        "mean_pred":        p[present].mean() if present.any() else 0.0,
        "count_MAE":        (p - t).abs().mean(),
        "count_bias":       (p - t).mean(),
    })
count_summary = pd.DataFrame(summary).set_index("class").round(2)
count_summary.to_csv(RUN_DIR / "per_image_count_summary.csv")
print(f"{len(per_image)} test images | CONF threshold = {CONF}")
count_summary

**Plot the counts.** (a) Each point is one test image: its true number of boxes (x-axis) against
the number the model detected (y-axis). Points on the dashed line are counted perfectly; a small
random jitter separates overlapping points. (b) For each class, the mean number of boxes per
image, true vs detected. The figure is also saved as `per_image_counts.png` in the run folder.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# (a) per-image total count: truth vs prediction (points on the dashed line = perfect count)
ax = axes[0]
mx = max(per_image["ALL_true"].max(), per_image["ALL_pred"].max()) + 1
jitter = lambda s: s + np.random.uniform(-0.15, 0.15, len(s))   # separate overlapping points
ax.scatter(jitter(per_image["ALL_true"]), jitter(per_image["ALL_pred"]), alpha=0.6)
ax.plot([0, mx], [0, mx], "k--", lw=1, label="perfect count")
ax.set_xlabel("true boxes per image"); ax.set_ylabel("detected boxes per image")
ax.set_title("(a) Symptomatic organs per test image"); ax.legend()

# (b) mean boxes per image, per class (only images that contain the class)
cs = count_summary.drop(index="ALL")
x = np.arange(len(cs))
axes[1].bar(x - 0.2, cs["mean_true"], 0.4, label="true")
axes[1].bar(x + 0.2, cs["mean_pred"], 0.4, label="detected")
axes[1].set_xticks(x); axes[1].set_xticklabels(cs.index, rotation=45, ha="right")
axes[1].set_ylabel("mean boxes per image"); axes[1].legend()
axes[1].set_title("(b) Mean count per image, by class")

plt.tight_layout(); plt.savefig(RUN_DIR / "per_image_counts.png", dpi=150); plt.show()

**Optional: effect of the confidence threshold.** The counts depend on `CONF`. The cell below
re-counts the test images at several thresholds. A low threshold over-counts (bias > 0), a high
one under-counts (bias < 0). The threshold where the bias is close to 0 is a sensible
operating point for counting.

In [ ]:
sweep = []
for th in [0.10, 0.25, 0.40, 0.55, 0.70]:
    pred_tot = np.array([(r.boxes.conf >= th).sum().item() for r in pred_results])
    if th < CONF:   # boxes below CONF were never kept, so re-predict at the lower threshold
        pred_tot = np.array([len(x.boxes) for x in model.predict(test_img_dir, conf=th, device=DEVICE,
                                                                 verbose=False, stream=True)])
    err = pred_tot - per_image["ALL_true"].values
    sweep.append({"conf_threshold": th, "count_MAE": np.abs(err).mean(), "count_bias": err.mean(),
                  "images_flagged": int((pred_tot > 0).sum())})
pd.DataFrame(sweep).round(2)

# 9. Log this experiment

Every time you run the notebook with different settings, the cell below adds **one row** to
`/content/runs/.../experiment_log.csv`. After a few runs, this table is the basis for your
*experiment* table in the report.

> **Colab deletes `/content` when the session ends.** Download the log, the run folder figures
> (`results.png`, confusion matrix) and `best.pt` (Files panel on the left, right-click,
> *Download*), or copy them to Google Drive.

In [ ]:
log_path = Path(PROJECT) / "experiment_log.csv"
entry = pd.DataFrame([{
    "run": RUN_DIR.name, "model": MODEL_NAME, "epochs_max": EPOCHS, "epochs_run": len(hist),
    "best_epoch": int(best_row["epoch"]), "imgsz": IMGSZ, "batch": BATCH, "patience": PATIENCE,
    "val_mAP50": round(val_metrics.box.map50, 3), "val_mAP50-95": round(val_metrics.box.map, 3),
    "test_precision": round(test_metrics.box.mp, 3), "test_recall": round(test_metrics.box.mr, 3),
    "test_mAP50": round(test_metrics.box.map50, 3), "test_mAP50-95": round(test_metrics.box.map, 3),
}])
log = pd.concat([pd.read_csv(log_path), entry]) if log_path.exists() else entry
log = log.drop_duplicates(subset="run", keep="last")
log.to_csv(log_path, index=False)
print("Saved", log_path)
log

# 10. Summary

In this notebook you:
- converted Pascal-VOC box annotations of the `Wilt` subset to YOLO format, correcting the
  image orientation, and made a reproducible 70/20/10 split;
- fine-tuned a pretrained YOLO detector on wilt symptoms;
- evaluated it with precision, recall and mAP on the validation and **test** sets, overall and
  per class;
- inspected predictions visually and logged the experiment.

**Ideas for improving results:** train longer (more `EPOCHS` with larger `PATIENCE`), use a larger
model (`yolo26s.pt`), use a larger `IMGSZ` for small symptoms, or (advanced) increase
augmentation. Watch out for over-fitting with this small dataset.

# Assignment: write your report

You now have everything you need. Your task is to **get the best results you can** by making
small changes to the settings in **Step 4.1** (e.g. `EPOCHS`, `MODEL_NAME`, `IMGSZ`), and then to
describe the study in a **written report of about 2 pages**. You will also present your findings
to the class in a short **PowerPoint presentation** built from the same material.

### Before you write
1. Run the notebook once with the default settings. This is your **baseline**.
2. Run **at least 2 more experiments**, changing **one setting at a time** so you can tell what
   caused any difference. Each run is added to `experiment_log.csv` (Step 9).
3. **Download** your log, figures and best weights before the Colab session ends (`/content`
   is wiped when the runtime disconnects).

### Report structure

**1. Introduction (~¼ page)**
- The plant-pathology problem: the disease, its cause and symptoms, and why it matters.
- Why automated, image-based diagnosis is useful (scale, speed, early detection).
- What object detection of **Wilt** symptoms in tomato means, in one or two sentences.
- **The objective of your study**, stated clearly in the last sentence of the introduction.

**2. Methodology (~½ page)**: give enough detail that someone else could repeat your work.
- **Dataset:** source, number of images, classes, number of instances per class, imbalance.
- **Pre-processing:** annotation/format conversion, EXIF orientation correction,
  train/validation/test split (70/20/10, random seed 42) and what each split is used for.
- **Model:** YOLO architecture and size, pretrained weights and **transfer learning**.
- **Training settings:** epochs, early-stopping patience, image size, batch size, hardware (GPU).
- **Evaluation metrics:** precision, recall, IoU, **mAP50** and **mAP50-95** (define each in one sentence).

**3. Results (~½-¾ page)**: report *what you observed*; leave the interpretation for the
discussion. You can use tables and figures, or describe the results in short sentences. Number every
table and figure, give each a caption, and refer to it in the text.

**4. Discussion and conclusion (~½ page)**: interpret your results. You do not have to answer all of these questions; use them as a guide:

- Which classes are detected well and which are not? Relate this to the **number of training
  boxes**, **object size**, and how **visually distinct** the classes are.
- At the **image level**, how often does the model notice the disease at all, and does it
  under- or over-count the symptomatic organs? Is it good enough to *flag* plants even if the box
  placement (mAP) is imperfect?
- What did changing the epochs / model size / image size do? Did a bigger model always help?
- Do the training curves show **over-fitting** (training loss keeps falling while validation
  mAP stalls or drops)? Did early stopping trigger?
- How different are the validation and test scores, and why might they differ?
- Look at the failure examples. Are they model errors, or questionable/missing annotations?
- **Limitations:** small dataset, class imbalance, a single random split. Photos taken on the
  same date often show the *same plant* (file names like `Gray_0509_001…010`), so near-duplicate
  images can land in both train and test and make the scores look better than they really are.
- Would this model be usable in a real greenhouse or field? What would you need to improve
  (more images, more varied conditions, balanced classes, a larger model, …)?

End with a short **conclusion** (2-3 sentences) that directly answers the objective stated in
your introduction, and suggest one or two concrete next steps.

**5. References**: cite the dataset, Ultralytics YOLO (https://docs.ultralytics.com) and any
plant-pathology sources you used for the introduction.

If anything in the notebook does not run, contact the instructors. We are happy to help.